# TD Learning: SARSA e *Q*-learning no Blackjack

#### Prof. Armando Alves Neto - Introdução ao Aprendizado por Reforço - PPGEE/UFMG

##### Alunos: Henrique Marques Cruz e Matheus Tadeu Alves de Carvalho

---

## Introdução

O objetivo deste notebook é treinar um agente para jogar Blackjack usando dois algoritmos clássicos de TD Learning: **SARSA** e ***Q*-learning**. A implementação utiliza a biblioteca `gymnasium` com o ambiente `Blackjack-v1`.

O Blackjack é um jogo de cartas amplamente usado como problema de referência em aprendizado por reforço (Sutton & Barto, Cap. 5). O objetivo é obter cartas cuja soma seja a maior possível sem ultrapassar 21, competindo contra o dealer.

As regras relevantes são:
- Cartas de figura (Valete, Dama, Rei) valem **10 pontos**
- O Ás vale **11** (ás *utilizável*) ou **1** — sempre se usa 11 enquanto não estourar
- Cartas numéricas valem seu **valor nominal**
- O dealer segue uma estratégia fixa: pede carta se sua soma for menor que 17, para caso contrário
- Se o jogador ultrapassar 21, ele perde imediatamente (*bust*)

## Formulação como MDP

Cada partida de Blackjack é um **episódio** de um MDP finito e episódico:

### Espaço de observações

O estado do jogo é descrito por uma tupla com 3 variáveis:

| Variável | Intervalo | Descrição |
|---|---|---|
| Soma do jogador | 4 – 21 | Soma atual das cartas do jogador |
| Carta visível do dealer | 1 – 10 | Valor da carta exposta do dealer (1 = Ás) |
| Ás utilizável | 0 ou 1 | 1 se o jogador possui um Ás contando como 11 |

O espaço de observações é `Tuple(Discrete(32), Discrete(11), Discrete(2))`, totalizando **704 estados** possíveis. Na prática, apenas os estados com soma entre 12 e 21 são alcançáveis com decisão (200 estados relevantes, conforme Sutton & Barto).

### Espaço de ações

O agente pode escolher entre duas ações a cada turno:
- **0 — Stick**: parar de pedir cartas
- **1 — Hit**: pedir mais uma carta

### Função de recompensa

As recompensas são dadas apenas ao final do episódio (sem desconto, $\gamma = 1$):
- Ganhar o jogo: **+1**
- Perder o jogo: **−1**
- Empate: **0**

### Término do episódio

O episódio termina quando:
1. O jogador pede carta e ultrapassa 21 (*bust*) → derrota imediata
2. O jogador decide parar (*stick*) → o dealer joga e o resultado é determinado

In [ ]:
import gymnasium as gym
import numpy as np
from functools import partial
from IPython.display import clear_output
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

## Codificação do espaço de estados

O ambiente retorna a observação como uma tupla `(soma_jogador, carta_dealer, as_utilizavel)`. Para armazenar a tabela $Q(s, a)$ como um array bidimensional, precisamos mapear cada tupla para um índice inteiro único.

Usamos a função `np.ravel_multi_index`, que converte um índice multidimensional em um índice plano:

$$
s = \text{soma} \times (11 \times 2) + \text{carta\_dealer} \times 2 + \text{as\_utilizavel}
$$

Por exemplo, o estado `(15, 7, 0)` (soma 15, dealer mostra 7, sem ás utilizável) é mapeado para o inteiro $15 \times 22 + 7 \times 2 + 0 = 344$.

Criando a classe para o TD learning.

In [ ]:
class TDlearning(object):

    def __init__(self, parameters):

        self.parameters = parameters

        # método a ser utilizado: 'SARSA' ou 'Q-learning'
        self.method = parameters['method']

        # contador de episódios rodados
        self.episode = 0

        # cria o ambiente Blackjack do gymnasium
        # natural=False: não dá recompensa extra por blackjack natural
        # sab=False:     não usa as regras exatas de Sutton & Barto
        self.env = gym.make('Blackjack-v1', natural=False, sab=False)

        # dimensões do espaço de observações: (32, 11, 2)
        # 32 valores possíveis para a soma do jogador (0–31)
        # 11 valores possíveis para a carta do dealer (0–10)
        #  2 valores possíveis para o ás utilizável (0 ou 1)
        self.obs_shape = (32, 11, 2)

        # número total de estados linearizados = 32 * 11 * 2 = 704
        self.num_states = np.prod(self.obs_shape)

        # número de ações disponíveis: 2 (stick=0, hit=1)
        self.num_actions = self.env.action_space.n

        # ------- parâmetros de aprendizado -------
        # γ: fator de desconto (1.0 = sem desconto, adequado para Blackjack)
        self.gamma = parameters['gamma']
        # ε: controla a exploração na política ε-soft
        #    ε alto → mais exploração; ε baixo → mais explotação
        self.eps   = parameters['eps']
        # α: taxa de aprendizado (quão rápido Q é atualizado)
        #    α alto → aprende rápido mas pode oscilar; α baixo → mais estável
        self.alpha = parameters['alpha']
        # -----------------------------------------

        # nome do arquivo para salvar/carregar a tabela Q
        # o prefixo diferencia os arquivos de cada método
        self.logfile = parameters['q-file']
        if self.method == 'SARSA':
            self.logfile = 'sarsa_' + self.logfile
        elif self.method == 'Q-learning':
            self.logfile = 'qlearning_' + self.logfile
        else:
            print('Método inválido! Use "SARSA" ou "Q-learning".')

        # inicializa (ou carrega) a tabela Q
        self.reset()

    ##########################################
    # converte a observação (tupla) em índice inteiro único
    def obs_to_state(self, obs):
        # obs = (soma_jogador, carta_dealer, as_utilizavel)
        # np.ravel_multi_index "achata" o índice multidimensional
        return int(np.ravel_multi_index(obs, self.obs_shape))

    ##########################################
    # inicializa a tabela Q e o ambiente
    def reset(self):
        # reseta o ambiente para o estado inicial
        self.env.reset()

        # Q(s, a) inicializado com zeros para todos os pares (estado, ação)
        # zeros são neutros: o agente não tem preferência inicial por nenhuma ação
        self.Q = np.zeros((self.num_states, self.num_actions))

        # se load_Q=True, tenta carregar uma tabela Q já treinada de um arquivo
        if self.parameters['load_Q']:
            try:
                with open(self.logfile, 'rb') as f:
                    data         = np.load(f)
                    self.Q       = data['Q']
                    self.episode = int(data['episodes'])
                print(f'Tabela Q carregada: {self.episode} episódios anteriores.')
            except:
                print('Arquivo não encontrado. Iniciando treinamento do zero.')

    ##########################################
    # retorna a política corrente como uma função parcial
    # copy=True: congela uma cópia de Q (útil para análise off-policy)
    # copy=False: usa referência direta a Q (política sempre atualizada)
    def curr_policy(self, copy=False):
        if copy:
            return partial(self.tabular_epsilon_soft_policy, np.copy(self.Q))
        else:
            return partial(self.tabular_epsilon_soft_policy, self.Q)

    ##########################################
    # persiste a tabela Q e o número de episódios em arquivo .npz
    def save(self):
        with open(self.logfile, 'wb') as f:
            np.savez(f, Q=self.Q, episodes=self.episode)

## Política $\varepsilon$-soft

A política $\varepsilon$-soft garante que todas as ações têm alguma probabilidade de serem escolhidas, o que mantém a **exploração** ao longo do treinamento. Sem exploração, o agente poderia ficar preso em políticas subótimas.

A ação gulosa (a que maximiza $Q(s, \cdot)$) é favorecida, mas as demais ações também são escolhidas com probabilidade mínima $\varepsilon/|\mathcal{A}|$:

$$
\pi(a|S_t) \gets 
                    \begin{cases}
                        1 - \varepsilon + \dfrac{\varepsilon}{|\mathcal{A}|},  & \text{se}~ a = \arg\max\limits_{a} Q(S_t,a) \quad\text{(explotação)},\\
                        \dfrac{\varepsilon}{|\mathcal{A}|}, & \text{caso contrário} \quad\text{(exploração).}
                    \end{cases}
$$

**Interpretação prática:** com $\varepsilon = 0.1$ e $|\mathcal{A}| = 2$, a ação gulosa é escolhida com probabilidade $0.95$ e a outra ação com probabilidade $0.05$.

In [ ]:
class TDlearning(TDlearning):
    ##########################################
    # seleciona uma ação segundo a política ε-soft
    def tabular_epsilon_soft_policy(self, Q, state):

        # probabilidade da ação gulosa (explotação)
        p_greedy = 1.0 - self.eps + self.eps / self.num_actions

        # probabilidade das demais ações (exploração)
        p_other  = self.eps / self.num_actions

        # monta o vetor de probabilidades para cada ação
        # a ação com maior Q(state, a) recebe p_greedy; as demais recebem p_other
        best_action = Q[state, :].argmax()
        prob = [p_greedy if a == best_action else p_other for a in range(self.num_actions)]

        # amostra uma ação de acordo com as probabilidades calculadas
        return np.random.choice(self.num_actions, p=np.array(prob))

## Método do SARSA

O SARSA é um algoritmo **on-policy**: ele atualiza $Q$ com base na ação $A'$ que a **mesma política** (com exploração) escolheria no próximo estado $S'$. O nome vem da quíntupla $(S, A, R, S', A')$ usada na atualização:

$$
Q(S,A) \gets Q(S,A) + \alpha \Big[\underbrace{R + \gamma\, Q(S',A')}_{\text{target}} - Q(S,A)\Big]
$$

O **target** é $R + \gamma\, Q(S',A')$, onde $A'$ é a ação **efetivamente escolhida** no próximo estado segundo $\pi$. Isso significa que o SARSA leva em conta a exploração ao aprender — se a política explorar em $S'$ e escolher uma ação subótima, esse custo se reflete em $Q$.

Passos de um turno:
1. Aplica ação $A$ no ambiente, observa $R$ e $S'$
2. Escolhe $A'$ a partir de $S'$ usando $\pi$ (com $\varepsilon$-soft)
3. Atualiza $Q(S,A)$ com o target $R + \gamma\, Q(S',A')$
4. Avança: $S \leftarrow S'$, $A \leftarrow A'$

In [ ]:
class TDlearning(TDlearning):
    ##########################################
    def sarsa(self, S, A):

        # aplica a ação A e observa o resultado do ambiente
        obs, R, terminated, truncated, _ = self.env.step(A)

        # o episódio termina por bust (terminated) ou por stick (truncated)
        done = terminated or truncated

        # converte a nova observação para índice de estado S'
        Sl = self.obs_to_state(obs)

        # escolhe A' a partir de S' usando a MESMA política π (on-policy)
        # isso distingue o SARSA do Q-learning: a atualização usa a ação real que será tomada
        Al = self.policy(Sl)

        # atualização SARSA:
        # erro TD = R + γ·Q(S', A') - Q(S, A)
        # o erro mede a diferença entre o que esperávamos e o que de fato ocorreu
        td_error = R + self.gamma * self.Q[Sl, Al] - self.Q[S, A]
        self.Q[S, A] = self.Q[S, A] + self.alpha * td_error

        # retorna o próximo estado, a próxima ação, a recompensa e o flag de término
        return Sl, Al, R, done

## Método do *Q*-learning

O *Q*-learning é um algoritmo **off-policy**: ele atualiza $Q$ usando o **máximo** sobre as ações do próximo estado, independentemente da política que o agente está seguindo. Isso faz com que o *Q*-learning aprenda diretamente a política ótima $\pi^*$, mesmo que o agente explore:

$$
Q(S,A) \gets Q(S,A) + \alpha \Big[\underbrace{R + \gamma \max_{a}\, Q(S',a)}_{\text{target}} - Q(S,A)\Big]
$$

O **target** é $R + \gamma \max_a Q(S', a)$: o melhor valor futuro *possível* a partir de $S'$, ignorando a exploração. Por isso, o *Q*-learning tende a convergir para a política ótima mais rapidamente do que o SARSA, mas pode ser mais instável em ambientes onde a exploração tem consequências (por exemplo, em cliffs o SARSA aprende um caminho mais seguro).

Passos de um turno:
1. Escolhe $A$ a partir de $S$ usando $\pi$ (com $\varepsilon$-soft)
2. Aplica $A$ no ambiente, observa $R$ e $S'$
3. Atualiza $Q(S,A)$ com o target $R + \gamma \max_a Q(S', a)$
4. Avança: $S \leftarrow S'$

In [ ]:
class TDlearning(TDlearning):
    ##########################################
    def qlearning(self, S):

        # escolhe a ação A a partir do estado atual usando π (ε-soft)
        A = self.policy(S)

        # aplica a ação A e observa o resultado do ambiente
        obs, R, terminated, truncated, _ = self.env.step(A)

        # o episódio termina por bust (terminated) ou por stick (truncated)
        done = terminated or truncated

        # converte a nova observação para índice de estado S'
        Sl = self.obs_to_state(obs)

        # atualização Q-learning:
        # erro TD = R + γ·max_a Q(S', a) - Q(S, A)
        # diferente do SARSA: usa o MÁXIMO sobre as ações de S' (off-policy)
        td_error = R + self.gamma * self.Q[Sl, :].max() - self.Q[S, A]
        self.Q[S, A] = self.Q[S, A] + self.alpha * td_error

        # retorna o próximo estado, a recompensa e o flag de término
        return Sl, R, done

## Execução de um episódio

Um episódio corresponde a uma partida completa de Blackjack. O loop roda até o episódio terminar — seja por *bust* ou por *stick*. A cada passo, o método correto (SARSA ou *Q*-learning) é chamado para atualizar a tabela $Q$.

**Diferença fundamental no loop:**
- No **SARSA**, a ação do próximo turno ($A'$) já é escolhida *dentro* do `sarsa()` e reutilizada no passo seguinte — a ação e o estado caminham juntos
- No ***Q*-learning**, a ação é escolhida no início de cada passo, sem precisar ser passada adiante

In [ ]:
class TDlearning(TDlearning):
    ##########################################
    def run_episode(self):

        # incrementa o contador de episódios
        self.episode += 1

        # obtém a política corrente (referência direta a Q, sempre atualizada)
        self.policy = self.curr_policy()

        # reinicia o ambiente e obtém a observação inicial
        obs, _ = self.env.reset()

        # converte a observação inicial para índice de estado
        S = self.obs_to_state(obs)

        # pré-seleciona a primeira ação (necessário para o SARSA,
        # que precisa do par (S, A) antes de interagir com o ambiente)
        A = self.policy(S)

        # acumula as recompensas do episódio
        rewards = []

        while True:

            if self.method == 'SARSA':
                # SARSA: passa (S, A), recebe (S', A') → A' será usado no próximo passo
                Sl, Al, R, done = self.sarsa(S, A)
                A = Al  # reutiliza a ação já escolhida para S'

            elif self.method == 'Q-learning':
                # Q-learning: escolhe A internamente a cada passo
                Sl, R, done = self.qlearning(S)

            else:
                print('Método errado.')
                break

            # avança para o próximo estado
            S = Sl

            # registra a recompensa do passo
            rewards.append(R)

            # verifica se o episódio terminou
            if done:
                break

        # salva a tabela Q ao final do episódio (se configurado)
        if self.parameters['save_Q']:
            self.save()

        # retorna a recompensa total do episódio (será +1, -1 ou 0 no Blackjack)
        return np.sum(np.array(rewards))

## Código principal

Parâmetros de configuração:

| Parâmetro | Descrição |
|---|---|
| `episodes` | Número total de episódios de treinamento |
| `gamma` | Fator de desconto $\gamma$ (1.0 = sem desconto) |
| `eps` | $\varepsilon$ da política $\varepsilon$-soft |
| `alpha` | Taxa de aprendizado $\alpha$ |
| `method` | Algoritmo: `'SARSA'` ou `'Q-learning'` |
| `save_Q` | Salva a tabela $Q$ em arquivo ao fim de cada episódio |
| `load_Q` | Carrega uma tabela $Q$ pré-treinada para continuar o treinamento |
| `q-file` | Nome base do arquivo da tabela $Q$ |

In [ ]:
if __name__ == '__main__':

    plt.ion()
    plt.rcParams['figure.figsize'] = (8, 3)
    plt.figure(1)

    # dicionário de parâmetros compartilhado entre os dois métodos
    parameters = {'episodes' : 50000,
                  'gamma'    : 1.0,    # sem desconto: recompensa terminal vale integralmente
                  'eps'      : 0.1,    # 10% de chance de explorar uma ação aleatória
                  'alpha'    : 0.01,   # taxa de aprendizado pequena para maior estabilidade
                  'method'   : '',     # será sobrescrito abaixo
                  'save_Q'   : True,
                  'load_Q'   : False,
                  'q-file'   : 'blackjack_qtable.npz',}

    # instancia um agente para cada método
    parameters['method'] = 'Q-learning'
    agent_q = TDlearning(parameters)

    parameters['method'] = 'SARSA'
    agent_s = TDlearning(parameters)

    # histórico de recompensas por episódio
    rewards_s = []
    rewards_q = []

    # médias móveis (janela de 500 episódios)
    avg_s = []
    avg_q = []

    # loop principal de treinamento
    for i in range(parameters['episodes']):

        # executa um episódio completo para cada agente
        rewards_s.append(agent_s.run_episode())
        rewards_q.append(agent_q.run_episode())

        # calcula a média móvel para suavizar a curva (reduz o ruído episódio a episódio)
        avg_s.append(np.mean(rewards_s[-500:]))
        avg_q.append(np.mean(rewards_q[-500:]))

        # plota a curva de aprendizado a cada 500 episódios
        if (i+1) % 500 == 0:
            clear_output(wait=True)
            plt.clf()

            # recompensas brutas (mais ruidosas) em traço fino e transparente
            plt.plot(rewards_s, 'r', alpha=0.15, linewidth=1)
            plt.plot(avg_s, 'r', linewidth=2, label='SARSA')
            #
            plt.plot(rewards_q, 'b', alpha=0.15, linewidth=1)
            plt.plot(avg_q, 'b', linewidth=2, label=r'$Q$-learning')
            #
            plt.axhline(0, color='gray', linewidth=0.8, linestyle='--')
            plt.legend(loc='upper left', bbox_to_anchor=(1, 1), fancybox=True, shadow=True, fontsize=12, facecolor='w')
            plt.xlabel('Episódios')
            plt.ylabel('Recompensa')
            plt.xlim([0, parameters['episodes']])
            plt.grid(alpha=0.3)
            plt.tight_layout()
            plt.gcf().patch.set_alpha(0)
            plt.show()
            plt.pause(0.1)

    print(f'\nTreinamento concluído!')
    print(f'Recompensa média final — SARSA:      {avg_s[-1]:.4f}')
    print(f'Recompensa média final — Q-learning: {avg_q[-1]:.4f}')

## Visualização da Política Aprendida

Após o treinamento, visualizamos a política gulosa $\pi(s) = \arg\max_a Q(s,a)$ como um heatmap. Cada célula indica se o agente deveria pedir carta (**Hit**) ou parar (**Stick**) naquele estado.

Os gráficos são separados em dois casos:
- **Sem Ás Utilizável**: o agente não possui um Ás contando como 11
- **Com Ás Utilizável**: o agente possui um Ás que pode contar como 11 sem estourar

A política de referência de Sutton & Barto é simples: *stick* se a soma for 20 ou 21, *hit* caso contrário.

In [ ]:
if __name__ == '__main__':

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # pares (rótulo, agente) para cada linha do gráfico
    method_agents = [('SARSA', agent_s), (r'$Q$-learning', agent_q)]
    titles_ace    = ['Sem Ás Utilizável', 'Com Ás Utilizável']

    for row, (method_label, agent) in enumerate(method_agents):
        for col, usable_ace in enumerate([0, 1]):
            ax = axes[row][col]

            # monta a grade de política:
            # linhas  = soma do jogador de 4 a 21 (18 valores)
            # colunas = carta visível do dealer de 1 a 10 (10 valores)
            policy_grid = np.zeros((18, 10))

            for player_sum in range(4, 22):
                for dealer_card in range(1, 11):
                    # codifica o estado como índice inteiro
                    s = agent.obs_to_state((player_sum, dealer_card, usable_ace))
                    # ação gulosa: argmax Q(s, ·)
                    policy_grid[player_sum - 4, dealer_card - 1] = agent.Q[s, :].argmax()

            # heatmap: azul = Hit (1), vermelho = Stick (0)
            ax.imshow(policy_grid, cmap='RdBu', vmin=0, vmax=1, aspect='auto',
                      origin='lower', extent=[0.5, 10.5, 3.5, 21.5])

            ax.set_title(f'{method_label} — {titles_ace[col]}', fontsize=12)
            ax.set_xlabel('Carta visível do dealer')
            ax.set_ylabel('Soma do jogador')
            ax.set_xticks(range(1, 11))
            ax.set_xticklabels(['A', 2, 3, 4, 5, 6, 7, 8, 9, 10])
            ax.set_yticks(range(4, 22))
            ax.grid(alpha=0.2)

    # legenda global
    hit_patch   = mpatches.Patch(color='steelblue', label='Hit (1)')
    stick_patch = mpatches.Patch(color='firebrick', label='Stick (0)')
    fig.legend(handles=[hit_patch, stick_patch], loc='lower center',
               ncol=2, fontsize=12, bbox_to_anchor=(0.5, -0.01))

    plt.suptitle('Política Aprendida — Blackjack', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.gcf().patch.set_alpha(0)
    plt.show()